Compiles 2014-2016 ACS income data and merges it to original BG-level dataset.

In [ ]:
import pandas as pd
import requests
import re
import zipfile
import io
import matplotlib.pyplot as plt

In [ ]:
data_folder = "" # path to data
figure_path = "" # path to figures folder location
ACS_path = ""

# Read final BG-level dataset

In [ ]:
# BG data
df_BG = pd.read_csv(data_folder + '23_level_BG_US_imputedquantiles_20250722.zip', index_col=0)
df_BG['STATEFP'] = df_BG['STATEFP'].astype(str).str.zfill(2)  # Ensure STATEFP is two digits
df_BG['COUNTYFP'] = df_BG['COUNTYFP'].astype(str).str.zfill(5)  # Ensure COUNTYFP is three digits
df_BG['BGFP'] = df_BG['BGFP'].astype(str).str.zfill(12)  # Ensure COUNTYFP is three digits
df_BG.set_index('BGFP', inplace=True)

print('Mean size of a BG:', df_BG[df_BG['total_pop_byBG']>0]['total_pop_byBG'].mean())
print('Std:', df_BG[df_BG['total_pop_byBG']>0]['total_pop_byBG'].std())

In [ ]:
# Median household income by BG
df_BG['median_household_income_byBG'].mean()

# Read ACS data

In [ ]:
# 2014 ACS data downloaded from https://data2.nhgis.org/main: B19013
df_ACS_2014 = pd.read_csv(ACS_path + 'nhgis0001_ds206_20145_blck_grp.csv', encoding='latin1')
df_ACS_2014['GEOID'] = df_ACS_2014['GEOID'].str.split('US').str[1]  # Remove decimal part from GEOID
df_ACS_2014.set_index('GEOID', inplace=True)
df_ACS_2014['ABDPE001'] = df_ACS_2014['ABDPE001'].str.replace('.', '')  # Replace missing values with NaN
df_ACS_2014['ABDPE001'] = pd.to_numeric(df_ACS_2014['ABDPE001'], errors='coerce') # df_ACS_2014['ABDPE001'].astype(float)  # Convert to float for analysis
df_ACS_2014['ABDPE001'].mean()

In [ ]:
# 2015 ACS data downloaded from https://data2.nhgis.org/main: B19013
df_ACS_2015 = pd.read_csv(ACS_path + 'nhgis0001_ds215_20155_blck_grp.csv', encoding='latin1')
df_ACS_2015['GEOID'] = df_ACS_2015['GEOID'].str.split('US').str[1]  # Remove decimal part from GEOID
df_ACS_2015.set_index('GEOID', inplace=True)
df_ACS_2015['ADNKE001'] = df_ACS_2015['ADNKE001'].str.replace('.', '')  # Replace missing values with NaN
df_ACS_2015['ADNKE001'] = pd.to_numeric(df_ACS_2015['ADNKE001'], errors='coerce') # df_ACS_2015['ADNKE001'].astype(float)  # Convert to float for analysis
df_ACS_2015['ADNKE001'].mean()

In [ ]:
# 2016 ACS data downloaded from https://data2.nhgis.org/main: B19013
df_ACS_2016 = pd.read_csv(ACS_path + 'nhgis0001_ds225_20165_blck_grp.csv', encoding='latin1')
df_ACS_2016['GEOID'] = df_ACS_2016['GEOID'].str.split('US').str[1]  # Remove decimal part from GEOID
df_ACS_2016.set_index('GEOID', inplace=True)
df_ACS_2016['AF49E001'] = df_ACS_2016['AF49E001'].str.replace('.', '')  # Replace missing values with NaN
df_ACS_2016['AF49E001'] = pd.to_numeric(df_ACS_2016['AF49E001'], errors='coerce') # df_ACS_2016['ADNKE001'].astype(float)  # Convert to float for analysis
df_ACS_2016['AF49E001'].mean()

# Merge ACS data

In [ ]:
cols_ACS = ['median_household_income_byBG_ACS2014', 'median_household_income_byBG_ACS2015', 'median_household_income_byBG_ACS2016']

In [ ]:
# Merge on FIPS
df_ACS = df_ACS_2014[['ABDPE001']].copy()
df_ACS.rename(columns={'ABDPE001': 'median_household_income_byBG_ACS2014'}, inplace=True)
df_ACS = df_ACS.merge(df_ACS_2015[['ADNKE001']], left_index=True, right_index=True, how='outer')
df_ACS.rename(columns={'ADNKE001': 'median_household_income_byBG_ACS2015'}, inplace=True)
df_ACS = df_ACS.merge(df_ACS_2016[['AF49E001']], left_index=True, right_index=True, how='outer')
df_ACS.rename(columns={'AF49E001': 'median_household_income_byBG_ACS2016'}, inplace=True)
df_ACS.head(3)

# Convert ACS 2010 > ACS 2020

In [ ]:
# Download crosswalk file to ACS_path
# https://secure-assets.ipums.org/nhgis/crosswalks/nhgis_bg2010_bg2020.zip

In [ ]:
# Read crosswalk file
df_crosswalk = pd.read_csv(ACS_path + 'nhgis_bg2010_bg2020\\nhgis_bg2010_bg2020.csv', dtype={'bg2010ge': str, 'bg2020ge': str, 'wt_pop': float})
print(f"  Crosswalk rows: {len(df_crosswalk):,}")

In [ ]:
# Reduce to needed columns
df_crosswalk = df_crosswalk[['bg2010ge', 'bg2020ge', 'wt_pop']].copy()
df_crosswalk.head(5)

In [ ]:
# Merge df_ACS with weights
df_ACS_weights = df_ACS.merge(df_crosswalk, left_index=True, right_on='bg2010ge', how='left')
print(f"  Merged rows: {len(df_ACS_weights):,}")
df_ACS_weights.head(5)

In [ ]:
# Compute weighted columns
for col in cols_ACS:
    df_ACS_weights[col + '_weighted'] = df_ACS_weights[col] * df_ACS_weights['wt_pop']
df_ACS_weights.head(5)

In [ ]:
# Remove rows with 0.0 weights
df_ACS_weights_nonzero = df_ACS_weights[df_ACS_weights['wt_pop'] > 0].copy()
print(f"  Non-zero weight rows: {len(df_ACS_weights_nonzero):,}")

In [ ]:
# Sum by bg2020ge
df_ACS_weighted_BG2020 = df_ACS_weights_nonzero.groupby('bg2020ge')[['wt_pop'] + [col + '_weighted' for col in cols_ACS]].sum(min_count=1)
df_ACS_weighted_BG2020.rename(columns={col + '_weighted': col for col in cols_ACS}, inplace=True)
for col in cols_ACS:
    df_ACS_weighted_BG2020[col] = df_ACS_weighted_BG2020[col] / df_ACS_weighted_BG2020['wt_pop']
df_ACS_weighted_BG2020.head(5)

# Merge to original BG data

In [ ]:
# Merge
df_BG_combined = df_BG.merge(df_ACS_weighted_BG2020, left_index=True, right_index=True, how='left')
df_BG_combined[['median_household_income_byBG','median_household_income_byBG_ACS2014', 'median_household_income_byBG_ACS2015', 'median_household_income_byBG_ACS2016']].head()

# Save data

In [ ]:
# Save
df_BG_combined.to_csv(data_folder + '23_level_BG_US_imputedquantiles_20260504.csv')